# Flag Algebras in Practice: Short tutorial

**Zászló** is a Python library implementing the *semidefinite method* for flag algebras.
Starting from a problem specification (which graphs are forbidden, what density to bound?), it
enumerates combinatorial structures, builds a semidefinite program (SDP), solves it, and
verifies the certificate in exact rational arithmetic.

In this tutorial, we'll use Mantel's theorem (surprise surprise) as a vehicle to show how to set up a Turán-type problem and interpret the results.

## Getting set up

Running the cell below will let us use the tools we need. It contains the barebones essentials for this library: click it, then press shift+enter to execute the cell. You'll need to do the same for all later code cells (they have that grey background, you can't miss 'em).

In [ ]:
from fractions import Fraction

from zaszlo import (
    FlagProblem,
    Hypergraph,
    build_flag_algebra_data,
    identify_sharps,
    round_certificate,
    solve_sdp,
    verify_certificate,
)

Okay, now, let's state our problem.

---
## 1. Mantel's Theorem

**Theorem (Mantel, 1907).** A triangle-free graph on $n$ vertices has at most $\lfloor n^2/4 \rfloor$
edges.  Equivalently, its edge density satisfies $p(K_2; G) \le \frac{1}{2}$, with equality for
the balanced complete bipartite graph $K_{\lfloor n/2\rfloor,\lceil n/2\rceil}$.

We will recover the bound $\frac{1}{2}$ via flag algebras working at order $n = 4$.

### Defining the Parameters

We'll first define the forbidden graph $K_3$: Some shorthand exists to make this graph, but this is the most general way to define a graph in Zászló.

In [ ]:
K3 = Hypergraph(3, 2, [(1, 2), (1, 3), (2, 3)])

Here, we constructed the graph by telling Zászló the number of vertices, uniformity of edges, and the edges themselves. For a little more information, run the cell below:

In [ ]:
K3

**Tip — self-describing objects.** Every Zászló object has a `.explain()` method returning a plain-text description (works in scripts, REPLs, and notebooks alike). In Jupyter, objects also render directly as annotated HTML cards — Jupyter picks this up automatically via `_repr_mimebundle_`. Try putting any object on the last line of a cell, or call `display(obj)`, to see the visual form. We'll be doing this throughout the tutorial!

Alright, now let's see how to encode our problem (maximize edges, forbid $K_3$) correctly:

In [ ]:
# Problem specification
prob_mantel = FlagProblem(
    4,              # n: admissible graphs on 4 vertices
    2,              # type_order: use types of order 0 and 2 (both even, like n=4)
    2,              # k: ordinary graphs (2-uniform)
    forbidden=[K3], # forbid non-induced K3
    minimize=False, # find an upper bound (minimize lambda)
)

Some quick notes: 

-`n` and `type_order` are variables you have some control over. Making these values larger will be more computationally intensive. `n` should probably be a single digit if you want the solver to ever find an answer.

-You also need $0\leq$ `type_order` $\leq$ `n`, with `type_order` and `n` of the same parity.

-Last, the format of `forbidden` is a list, and you can feed (comma-separated) entries to forbid more graphs.

As above, run the following cell to display some info about our problem:

In [ ]:
prob_mantel

### Building Flags

We now need to take these constraints and generate some flags! The next two cells should help with that. This object is quite rich, check out the longer tutorial to see more.

In [ ]:
data_mantel = build_flag_algebra_data(prob_mantel)

In [ ]:
print(data_mantel.explain())

### Solving the SDP

`solve_sdp` uses Clarabel to find the optimal $\lambda$ and PSD matrices $\{Q_\sigma\}$.
`extract_Q=True` stores the certificate matrices so we can verify them afterwards. Run the following cell to solve the SDP. It should be pretty fast here, since this problem is small.

In [ ]:
result_mantel = solve_sdp(data_mantel, extract_Q=True)

print(result_mantel.explain())

### Sharp (extremal) graphs

An admissible graph $H$ is *sharp* when its SDP slack is zero: the bound is tight at $H$.
Sharps are the graphs that appear in the extremal construction — for Mantel, the extremal
graphon is the balanced bipartite graphon ($K_{n/2,n/2}$ in the limit).

In [ ]:
sharps_mantel = identify_sharps(data_mantel, result_mantel)


sharps_mantel

## Getting an Exact bound

We aren't done quite yet though: due to floating-point arithmetic errors, inaccuracies may arise. What follows is a "round and verify" process that results in a complete computer-assisted proof.

Run the following cell, which should return `valid:False.' It's grumpy that some residuals are negative--but those residuals are numerically. We'll round this and the residuals in a particular fashion that yields a slightly weaker, but certifiable, bound.

In [ ]:
cert_raw = verify_certificate(data_mantel, result_mantel)
print(f"valid          : {cert_raw['valid']}")
print(f"min_residual   : {float(cert_raw['min_residual']):.2e}  ← may be slightly negative")
print(f"min_psd_eigval : {cert_raw['min_psd_eigval']:.2e}")
print(f"lam_certified  : {cert_raw['lam_certified']}")

Run the below. See that the rounded cert is verified?

In [ ]:
result_mantel_rat = round_certificate(data_mantel, result_mantel, denom_limit=1000)
cert_rat = verify_certificate(data_mantel, result_mantel_rat)

cert_rat

Here ends our brief tutorial! This should get you started with any Turán-type problem you fancy. The longer tutorial will also walk you through using Zászló to find the maximum induced density of subgraphs beyond just the edge via the Pentagon problem, as well how to work with hypergraphs of higher uniformity. It also has more pictures!